In [1]:
import pandas as pd

# Step 1: read as cp1252 and drop exact duplicates
df = pd.read_csv("School_Data_CSV.csv", dtype=str, encoding="cp1252")
df = df.drop_duplicates()

# Step 2: keep valid nine-digit EMIS numbers, set the rest aside
valid_id = df["emisno"].str.fullmatch(r"\d{9}", na=False)
set_aside = df[~valid_id]
df = df[valid_id].copy()

# Step 3: standardise quintile to integers 1 to 5
q = df["quintile"].str.strip().str.upper().str.extract(r"^N?Q?([1-5])$")[0]
df["quintile_clean"] = pd.to_numeric(q)

# Step 4: coerce coordinates, then apply South African range checks
df["lat"] = pd.to_numeric(df["new_lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["new_long"], errors="coerce")
ok = df["lat"].between(-35, -22) & df["lon"].between(16, 33)
df.loc[~ok, ["lat", "lon"]] = float("nan")

# Step 5: keep eight-digit ward codes only
df["ward_clean"] = df["ward_id"].where(df["ward_id"].str.fullmatch(r"\d{8}", na=False))

# Step 2 (cont.): one row per school, most frequent value, ties take the lowest
def mode_low(s):
    s = s.dropna()
    return s.value_counts().sort_index().idxmax() if len(s) else float("nan")

schools = df.groupby("emisno").agg(
    quintile=("quintile_clean", mode_low),
    ward=("ward_clean", mode_low),
    lat=("lat", "first"), lon=("lon", "first"))

print("Schools:", len(schools))
print("Usable quintile:", schools["quintile"].notna().sum())
print("Eight-digit ward:", schools["ward"].notna().sum())
print("Valid coordinate pair:", schools[["lat", "lon"]].notna().all(axis=1).sum())
print("Rows set aside (invalid EMIS):", len(set_aside))

Schools: 33054
Usable quintile: 25048
Eight-digit ward: 27634
Valid coordinate pair: 27487
Rows set aside (invalid EMIS): 2523
